# AfroViewz 02: Cleaning, de-duplication and a leak free split

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/02_data_cleaning_and_split.ipynb)

**Workflow stage:** data preparation.

The v1 capstone used the train, valid and test split that came with one Roboflow download. Public pothole datasets are full of repeats: the same photo uploaded to several projects, augmented copies of one photo, and consecutive frames of one drive. When a near copy of a test image sits in the training set, the test score measures memory, not skill.

This notebook:
1. merges the cleaned sources from notebook 01,
2. finds near duplicate images with perceptual hashing, inside and across sources,
3. drops exact repeats and ties near repeats into one group,
4. splits **by group** (80 / 10 / 10 per source), so no group spans two splits,
5. checks the split for leakage and exports it in the layout RF-DETR reads.

The Ugandan Mapillary photos are left out on purpose: they are a holdout, labelled in notebook 04 and never used for training.

## 1. Setup

In [ ]:
# Works on Colab and locally. On Colab: mounts Drive, gets the code, installs it.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Capstone"):
        subprocess.run(["git", "clone", "-q", "-b", "afroviewz", "https://github.com/JBWandera/Capstone.git", "/content/Capstone"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/Capstone", "pull", "-q"], check=True)
    os.chdir("/content/Capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"], check=True)
    sys.path.insert(0, "/content/Capstone/src")
from afroviewz import paths
print("data root:", paths.data_root())


In [ ]:
import json
import pandas as pd
from afroviewz.data import build, coco as C, dedup, split

TRAIN_SOURCES = ["stellenbosch", "roboflow_seg", "capstone_v1"]
DATASET = "afroviewz-v1"
OUT = paths.processed_dir(DATASET)
interim = {name: paths.interim_dir(name) for name in TRAIN_SOURCES}
for name, d in interim.items():
    print(name, C.summary(C.load(d / "coco.json")))


## 2. How perceptual hashing finds near duplicates

A pHash shrinks an image to 32x32 grey pixels, takes the discrete cosine transform and keeps the sign pattern of the 64 lowest frequencies. Resizing, recompression, small colour shifts and small crops barely change those 64 bits, so two images whose hashes differ in 6 bits or fewer (Hamming distance) are treated as the same scene.

Comparing every pair of 10,000+ images is 50 million comparisons. Instead the 64 bits are cut into 8 bands of 8 bits: two hashes within distance 7 must match exactly on at least one band, so only images sharing a band are compared.

## 3. Build the dataset

In [ ]:
report = build.build_from_interim(interim, OUT, dup_distance=6, seed=42)
print(json.dumps(report["dedup"], indent=2))


## 4. What de-duplication found

In [ ]:
allc = C.load(OUT / "all.coco.json")
imgs = pd.DataFrame(allc["images"])
clusters = imgs.groupby("dup_cluster").agg(n=("id", "size"), sources=("source", lambda s: ",".join(sorted(set(s)))))
multi = clusters[clusters.n > 1]
print(f"{len(multi)} near duplicate clusters holding {multi.n.sum()} images")
print("clusters that span more than one source:", (multi.sources.str.contains(",")).sum())
multi.sources.value_counts().head(10)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_cluster_examples(k=4, per=4):
    if multi.empty:
        print('No near duplicates found.')
        return
    k = min(k, len(multi))
    big = multi.sort_values("n", ascending=False).head(k).index
    fig, axes = plt.subplots(k, per, figsize=(3.2 * per, 2.6 * k), squeeze=False)
    for r, cid in enumerate(big):
        rows = imgs[imgs.dup_cluster == cid].head(per)
        for c in range(per):
            ax = axes[r, c]
            ax.set_axis_off()
            if c < len(rows):
                im = rows.iloc[c]
                ax.imshow(Image.open(OUT / "images" / im.file_name))
                ax.set_title(f"{im.source}\n{im.split}", fontsize=8)
    plt.suptitle("Near duplicate clusters: each row is one scene")
    plt.tight_layout(); plt.show()

show_cluster_examples()


## 5. The split

Each row of a cluster above sits in one split only. The table below shows images and boxes per split and source.

In [ ]:
anns = pd.DataFrame(allc["annotations"])
box_counts = anns.groupby("image_id").size()
imgs["boxes"] = imgs.id.map(box_counts).fillna(0).astype(int)
table = imgs.groupby(["split", "source"]).agg(images=("id", "size"), boxes=("boxes", "sum"),
                                              pothole_free=("boxes", lambda b: (b == 0).sum()))
table


In [ ]:
problems = split.check_no_leakage(allc, {s: imgs[imgs.split == s].id.tolist() for s in imgs.split.unique()})
assert not problems, problems[:5]
cross = imgs.groupby("dup_cluster").split.nunique()
assert (cross <= 1).all()
print("No group and no near duplicate cluster spans two splits.")


## 6. Compare with the v1 capstone split

How much of the old v1 test set had a near copy in the old v1 training set? This is the leakage the v1 numbers carried.

In [ ]:
v1 = imgs[imgs.source == "capstone_v1"]
old_test = v1[v1.orig_split == "test"]
old_train_clusters = set(v1[v1.orig_split == "train"].dup_cluster)
leaked = old_test.dup_cluster.isin(old_train_clusters)
print(f"v1 test images with a near copy in v1 train: {leaked.sum()} of {len(old_test)} ({leaked.mean():.0%})")


## Outputs

`processed/afroviewz-v1/` on Drive:
* `rfdetr/train|valid|test/_annotations.coco.json` with the images, ready for RF-DETR
* `all.coco.json` with every image's source, group, near duplicate cluster and split
* `build_report.json`

**Next:** notebook 03, exploratory data analysis.